# PCam CNN Training on Google Colab

This notebook trains a CNN using train/validation/test splits.
**Prerequisite:** Upload `train_new.zip`, `val_new.zip`, `test_new.zip` and the three CSV files to Google Drive.

## 1. Clone Repository

In [ ]:
!git clone https://github.com/goodguyjuro/pcam-cancer-detection.git
%cd pcam-cancer-detection
!pwd
!ls -la

## 2. Install Dependencies

In [ ]:
!pip install -q torch torchvision numpy matplotlib pandas scikit-learn

## 3. Mount Google Drive and Copy Split Files

Update `DRIVE_PATH` to match where you saved them.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/My Drive/pcam_splits'
DATA_DIR = '/content/pcam-cancer-detection/data'
!mkdir -p "$DATA_DIR"
!cp "$DRIVE_PATH/train_new.zip" "$DATA_DIR/"
!cp "$DRIVE_PATH/val_new.zip" "$DATA_DIR/"
!cp "$DRIVE_PATH/test_new.zip" "$DATA_DIR/"
!cp "$DRIVE_PATH/labels_train.csv" "$DATA_DIR/"
!cp "$DRIVE_PATH/labels_val.csv" "$DATA_DIR/"
!cp "$DRIVE_PATH/labels_test.csv" "$DATA_DIR/"
!ls -lh "$DATA_DIR"

## 4. Unzip the Split Files

In [ ]:
%cd /content/pcam-cancer-detection/data

!unzip -q train_new.zip -d train_new_tif
!unzip -q val_new.zip -d val_new_tif
!unzip -q test_new.zip -d test_new_tif

from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
import shutil

def convert_tif_folder_to_png(src_dir, dst_dir):
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    tif_files = list(src_dir.rglob("*.tif")) + list(src_dir.rglob("*.tiff"))

    for p in tqdm(tif_files, desc=f"Converting {src_dir.name}"):
        out_path = dst_dir / (p.stem + ".png")
        if out_path.exists():
            continue

        with Image.open(p) as img:
            img = img.convert("RGB")
            img.save(out_path, format="PNG", optimize=False)

convert_tif_folder_to_png("train_new_tif", "train_new")
convert_tif_folder_to_png("val_new_tif", "val_new")
convert_tif_folder_to_png("test_new_tif", "test_new")

# Optional: remove TIFF folders to save Colab disk space
shutil.rmtree("train_new_tif")
shutil.rmtree("val_new_tif")
shutil.rmtree("test_new_tif")

!ls -lh
%cd /content/pcam-cancer-detection

## 5. Verify Split Data Structure

In [ ]:
from pathlib import Path

data_dir = Path("/content/pcam-cancer-detection/data")

print("Train PNG:", len(list((data_dir / "train_new").glob("*.png"))))
print("Val PNG:", len(list((data_dir / "val_new").glob("*.png"))))
print("Test PNG:", len(list((data_dir / "test_new").glob("*.png"))))

print("Labels train exists:", (data_dir / "labels_train.csv").exists())
print("Labels val exists:", (data_dir / "labels_val.csv").exists())

## 6. Train Baseline Model (v1)

In [ ]:
!python /content/pcam-cancer-detection/scripts/train.py \
  --model_version 1 \
  --epochs 5 \
  --batch_size 128 \
  --num_workers 2 \
  --amp \
  --data_dir /content/pcam-cancer-detection/data

## 7. Train Augmentation Model (v2)

In [ ]:
!python /content/pcam-cancer-detection/scripts/train.py --model_version 2 --epochs 5 --batch_size 64 --num_workers 1 --data_dir /content/pcam-cancer-detection/data

## 8. Train Dropout Model (v3)

In [ ]:
!python /content/pcam-cancer-detection/scripts/train.py --model_version 3 --epochs 5 --batch_size 64 --num_workers 1 --data_dir /content/pcam-cancer-detection/data

## 9. View Results

In [ ]:
!ls -lh results/
!find results -type f -name '*.png'